# 05 — Bitki Önerisi

Her lokasyon × yıl kombinasyonu için modelin tahmin ettiği en yüksek verimli bitkiyi öner.

**Strateji:** Normal senaryo, full-season baseline modeli  
**Çıktı:** Her grid noktası için önerilen bitki ve beklenen verim

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
from pathlib import Path

from src.data.loader import load_season_agg

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path('../data/processed')
OUTPUTS   = Path('../outputs')
FIGS      = '../outputs/figures/'

## 1. Model ve veriyi yükle

In [ ]:
model = joblib.load(OUTPUTS / 'models/baseline_lgbm_fullseason.joblib')
print('Model yüklendi.')

agg = load_season_agg()
print(f'Toplam satır: {len(agg):,}')

## 2. Normal senaryo, başarılı hasatlar

In [ ]:
df = agg[(agg['wav_scenario'] == 'normal') & (agg['sim_success'] == 1)].copy()
print(f'Normal senaryo, başarılı hasat: {len(df):,} satır')
print(f'Lokasyon sayısı: {df[["latitude", "longitude"]].drop_duplicates().shape[0]}')
print(f'Bitki sayısı: {df["crop_name"].nunique()}')

## 3. Target encoding (train verisiyle tutarlı)

In [ ]:
import json

TARGET = 'harvest_twso'

with open(PROCESSED / 'crop_te_map.json') as f:
    crop_te_export = json.load(f)

global_mean = crop_te_export.pop('__global_mean__')
crop_te_map = crop_te_export  # {crop_name: mean_yield}

df['crop_te'] = df['crop_name'].map(crop_te_map).fillna(global_mean)
print('crop_te eklendi (02 notebook ile aynı mapping).')
print(df[['crop_name', 'crop_te']].drop_duplicates().sort_values('crop_te').to_string())

## 4. Model tahmini

In [ ]:
FEATURE_COLS = [
    'mean_temp', 'total_precip', 'mean_humidity', 'mean_rftra',
    'max_lai', 'max_tagp', 'max_dvs', 'season_days',
    'latitude', 'longitude', 'elevation', 'year', 'WAV', 'crop_te',
]

df['predicted_twso'] = model.predict(df[FEATURE_COLS])
print(f'Tahmin tamamlandı. Örnek:')
df[['latitude', 'longitude', 'crop_name', 'harvest_twso', 'predicted_twso']].head()

## 5. Her lokasyon × yıl için en iyi bitkiyi seç

In [ ]:
best = (
    df.loc[df.groupby(['latitude', 'longitude', 'year'])['predicted_twso'].idxmax()]
    [['latitude', 'longitude', 'year', 'crop_name', 'predicted_twso', 'harvest_twso']]
    .reset_index(drop=True)
)

print(f'Öneri sayısı: {len(best):,}')
print('\nEn sık önerilen bitkiler:')
print(best['crop_name'].value_counts())

## 6. Harita — en son yıl için önerilen bitki

In [ ]:
last_year = best['year'].max()
map_df = best[best['year'] == last_year].copy()

crops = sorted(map_df['crop_name'].unique())
palette = sns.color_palette('tab20', len(crops))
color_map = dict(zip(crops, palette))

fig, ax = plt.subplots(figsize=(13, 7))
for crop, grp in map_df.groupby('crop_name'):
    ax.scatter(grp['longitude'], grp['latitude'],
               color=color_map[crop], s=200, marker='s',
               edgecolors='gray', linewidths=0.3, label=crop)

ax.set_xlabel('Boylam')
ax.set_ylabel('Enlem')
ax.set_title(f'Önerilen Bitki — {last_year} (Normal Senaryo)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(FIGS + 'crop_recommendation_map.png', bbox_inches='tight')
plt.show()

## 7. Bölge bazında öneri tutarlılığı (yıllar arası)

In [ ]:
# Her lokasyon için en sık önerilen bitki ve tutarlılık oranı
consistency = (
    best.groupby(['latitude', 'longitude', 'crop_name'])
    .size().reset_index(name='count')
    .sort_values('count', ascending=False)
)
total_years = best['year'].nunique()
top_crop = (
    consistency.loc[consistency.groupby(['latitude', 'longitude'])['count'].idxmax()]
    .assign(consistency=lambda d: d['count'] / total_years)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sol: en sık önerilen bitki
for crop, grp in top_crop.groupby('crop_name'):
    axes[0].scatter(grp['longitude'], grp['latitude'],
                    color=color_map.get(crop, 'gray'), s=200, marker='s',
                    edgecolors='gray', linewidths=0.3, label=crop)
axes[0].set_title('En Sık Önerilen Bitki (Tüm Yıllar)')
axes[0].set_xlabel('Boylam')
axes[0].set_ylabel('Enlem')
axes[0].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

# Sağ: tutarlılık oranı
sc = axes[1].scatter(top_crop['longitude'], top_crop['latitude'],
                     c=top_crop['consistency'], cmap='RdYlGn',
                     s=200, marker='s', edgecolors='gray', linewidths=0.3,
                     vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label='Tutarlılık oranı')
axes[1].set_title('Yıllar Arası Öneri Tutarlılığı')
axes[1].set_xlabel('Boylam')
axes[1].set_ylabel('Enlem')

plt.tight_layout()
plt.savefig(FIGS + 'crop_recommendation_consistency.png', bbox_inches='tight')
plt.show()

## 8. Kaydet

In [ ]:
best.to_parquet(OUTPUTS / 'crop_recommendations.parquet', index=False)
top_crop.to_parquet(OUTPUTS / 'crop_recommendations_consistency.parquet', index=False)
print('Kaydedildi.')
print(best.head())